# Week 5 · Day 3 (Part 2) — The Same Linear Regression, Now with a Class

In the last notebook we trained a linear-regression model completely by hand — predict, measure error, compute gradients, nudge `m` and `b`, repeat. It worked, but the code was scattered across many cells, and the important values (`m`, `b`, the learning rate, the history) were loose variables floating around.

Today we do **exactly the same maths**, but we organize it neatly using **Object-Oriented Programming (OOP)** — we build a `LinearRegression` *class*. Nothing about the learning changes. We are only tidying the code into one self-contained object.

**Why bother?** Because this is how every real machine-learning library is built. When you later write `model = LinearRegression()` in scikit-learn, you are creating an object exactly like the one we build here. By the end you'll understand what that line really does.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## 1. A quick reminder: what is a class?

A **class** is a blueprint that bundles together:
- **data** (called *attributes*) — the things the object remembers, and
- **actions** (called *methods*) — the things the object can do.

An **object** is one specific thing built from that blueprint.

Think of a car blueprint (the class) versus an actual car (the object). The blueprint says "a car *has* a speed and *can* accelerate." Each real car has its own speed.

For our model:
- the **attributes** it remembers are the slope `m`, the intercept `b`, the learning rate, and the training history;
- the **methods** it can do are `predict`, compute `cost`, and `train_one_epoch`.

Everything we wrote loosely in the last notebook now lives inside one object.

---
## 2. The data (unchanged)

Same tiny dataset as before — study hours vs exam marks — so we can confirm we get the identical answers.

In [ ]:
x = np.array([1, 2, 3, 4, 5], dtype=float)   # hours studied
y = np.array([3, 5, 6.5, 9, 10.5])           # marks

plt.scatter(x, y, color="red", s=80, zorder=3)
plt.xlabel("hours studied"); plt.ylabel("marks")
plt.title("The same 5 data points"); plt.grid(True, alpha=0.3)
plt.show()

---
## 3. Building the `LinearRegression` class, piece by piece

We'll construct the class in stages so you see how each part maps to code you already wrote by hand. Here is the full class; below it, we explain every method.

In [ ]:
class LinearRegression:
    """A simple linear regression model, y = m*x + b, trained by gradient descent."""

    def __init__(self, learning_rate=0.02):
        # __init__ runs once when the object is created. It sets up the starting state.
        self.m = 0.0                 # slope        (starts at a bad guess)
        self.b = 0.0                 # intercept    (starts at a bad guess)
        self.lr = learning_rate      # step size
        self.history = []            # remembers (m, b, cost) after each epoch

    def predict(self, x):
        # The model itself: a straight line.
        return self.m * x + self.b

    def cost(self, x, y):
        # How wrong the current line is: average squared error.
        errors = self.predict(x) - y
        return np.mean(errors ** 2) / 2

    def _gradients(self, x, y):
        # Which way is downhill for m and b. (The leading _ means "internal helper".)
        errors = self.predict(x) - y
        grad_m = np.mean(errors * x)
        grad_b = np.mean(errors)
        return grad_m, grad_b

    def train_one_epoch(self, x, y):
        # ONE step of learning: the four moves, now in a single method call.
        grad_m, grad_b = self._gradients(x, y)
        self.m = self.m - self.lr * grad_m
        self.b = self.b - self.lr * grad_b
        self.history.append((self.m, self.b, self.cost(x, y)))
        return self.m, self.b, self.cost(x, y)

### What each part is, in plain words

- **`self`** — this word means "this particular object." `self.m` is *this model's* slope. It's how an object refers to its own attributes.
- **`__init__`** — the *constructor*. It runs automatically the moment you create the object, and sets the starting values. This replaces the loose `m = 0.0; b = 0.0; alpha = 0.02` lines from before.
- **`predict`, `cost`, `_gradients`** — the exact same three functions from the manual notebook, now living inside the object. The only change: they read `self.m` and `self.b` instead of taking `m` and `b` as arguments, because the object already remembers them.
- **`train_one_epoch`** — the four moves (predict → error → gradients → update), wrapped into one call. This is the method we'll call once per epoch, just like before.

Notice: **no new maths.** Every formula is identical. We've only *organized* it.

---
## 4. Creating the model object

Now we build one actual model from the blueprint. This single line runs `__init__`, setting `m=0`, `b=0`, and the learning rate.

In [ ]:
model = LinearRegression(learning_rate=0.02)

print("A model object now exists.")
print(f"  starting slope     m = {model.m}")
print(f"  starting intercept b = {model.b}")
print(f"  learning rate         = {model.lr}")
print(f"  starting cost         = {model.cost(x, y):.3f}")

`model` is now an object that *remembers its own* `m`, `b`, and learning rate. We no longer pass those around by hand — the object holds them. Let's train it, one epoch at a time, exactly as before.

---
## 5. Epoch 1 — one method call

In the manual notebook, one epoch took several lines. Now it's a single call: `model.train_one_epoch(x, y)`. The object does the four moves internally and updates its own `m` and `b`.

In [ ]:
# --- EPOCH 1 ---
new_m, new_b, new_cost = model.train_one_epoch(x, y)

print(f"after epoch 1:  m = {new_m:.4f}   b = {new_b:.4f}   cost = {new_cost:.4f}")

In [ ]:
def show_line(model, x, y, title=""):
    plt.scatter(x, y, color="red", s=80, zorder=3, label="data")
    x_line = np.linspace(0, 6, 100)
    plt.plot(x_line, model.predict(x_line), color="blue", linewidth=2,
             label=f"y = {model.m:.3f}x + {model.b:.3f}")
    plt.xlabel("hours studied"); plt.ylabel("marks")
    plt.title(title); plt.legend(); plt.grid(True, alpha=0.3); plt.ylim(0, 13)
    plt.show()

show_line(model, x, y, title="After Epoch 1")

Same result as the manual notebook: slope ≈ 0.484, cost dropped from 26.75 to ≈ 15.6. The object updated *itself* — `model.m` and `model.b` now hold the new values.

---
## 6. Epochs 2 and 3 — the same one-line call

Because the object remembers its own state, each epoch is now the identical single line. We don't re-supply `m` and `b`; the model already has them.

In [ ]:
# --- EPOCH 2 ---
model.train_one_epoch(x, y)
print(f"after epoch 2:  m = {model.m:.4f}   b = {model.b:.4f}   cost = {model.cost(x, y):.4f}")
show_line(model, x, y, title="After Epoch 2")

In [ ]:
# --- EPOCH 3 ---
model.train_one_epoch(x, y)
print(f"after epoch 3:  m = {model.m:.4f}   b = {model.b:.4f}   cost = {model.cost(x, y):.4f}")
show_line(model, x, y, title="After Epoch 3")

The line is climbing toward the points, exactly as before — but now the code doing it is far cleaner. This is the payoff of OOP: the object carries its own state, so calling it repeatedly is trivial.

---
## 7. Training to completion with a loop

Since one epoch is now a single method call, running many epochs is a tidy loop. We create a **fresh** model first (so we start clean from epoch 0), then train it fully.

In [ ]:
model = LinearRegression(learning_rate=0.02)   # a brand-new model, back to m=0, b=0

for epoch in range(1, 51):
    model.train_one_epoch(x, y)
    if epoch in (1, 5, 10, 20, 30, 50):
        print(f"epoch {epoch:2d}:  m = {model.m:.3f}   b = {model.b:.3f}   cost = {model.cost(x, y):.4f}")

In [ ]:
show_line(model, x, y, title="After 50 epochs — the fitted line")

Identical final answer to the manual notebook (`y ≈ 2.02x + 0.65`), because it *is* the same maths — just organized into an object.

---
## 8. The object remembered its whole journey

Because we stored every step inside `self.history`, the object carries its own training record. We can pull it straight out of the model to plot the cost curve and the sweep of lines — no separate bookkeeping needed.

In [ ]:
# history is a list of (m, b, cost) tuples, one per epoch — stored inside the object
costs = [h[2] for h in model.history]

plt.plot(range(1, len(costs) + 1), costs, color="purple", linewidth=2)
plt.xlabel("epoch"); plt.ylabel("cost")
plt.title("The cost falls as the model learns")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.scatter(x, y, color="red", s=80, zorder=3, label="data")
x_line = np.linspace(0, 6, 100)

for i, (mm, bb, cc) in enumerate(model.history):
    shade = (i + 1) / len(model.history)
    plt.plot(x_line, mm * x_line + bb, color="blue",
             alpha=0.15 + 0.85 * shade, linewidth=1)

plt.xlabel("hours studied"); plt.ylabel("marks")
plt.title("Every epoch's line — faint (early) to bold (final)")
plt.ylim(0, 13); plt.grid(True, alpha=0.3)
plt.show()

---
## 9. Using the trained model

Prediction is now a clean method call on the object.

In [ ]:
hours = 3.5
print(f"Trained model:  y = {model.m:.3f} x + {model.b:.3f}")
print(f"A student studying {hours} hours is predicted to score {model.predict(hours):.1f} marks")

---
## 10. The real payoff: reuse

Here's what OOP buys us that loose variables couldn't. Because the model is a self-contained blueprint, we can create **several independent models** and compare them — for example, to see how the **learning rate** affects training. Each object keeps its own separate `m`, `b`, and history without interfering with the others.

In [ ]:
# Train three separate models with different learning rates
results = {}
for rate in [0.005, 0.02, 0.05]:
    mdl = LinearRegression(learning_rate=rate)     # a fresh, independent object
    for _ in range(50):
        mdl.train_one_epoch(x, y)
    results[rate] = [h[2] for h in mdl.history]    # this model's cost history

# Compare how fast each one learned
for rate, costs in results.items():
    plt.plot(range(1, 51), costs, linewidth=2, label=f"learning rate = {rate}")

plt.xlabel("epoch"); plt.ylabel("cost")
plt.title("Same model, three learning rates — the power of reusable objects")
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

Each line is a *separate model object* that trained independently. The slowest learning rate (0.005) crawls down; the fastest here (0.05) drops quickest. Writing this comparison with loose variables would have been a tangle of `m1, b1, m2, b2, ...`. With a class, it's a clean three-line loop.

*(A learning rate can also be set too high — try `0.3` and watch the cost blow up instead of falling. That failure is worth seeing once.)*

---
## Summary — OOP didn't change the maths, it organized it

- A **class** is a blueprint bundling **data (attributes)** with **actions (methods)**.
- Our `LinearRegression` object *remembers* its own `m`, `b`, learning rate, and history (attributes), and can `predict`, compute `cost`, and `train_one_epoch` (methods).
- **`__init__`** sets the starting state; **`self`** lets the object refer to its own data.
- Every formula is **identical** to the manual notebook — we only moved the code inside an object, so the numbers match exactly.
- The reward is **cleanliness and reuse**: one call per epoch, the object carries its own history, and we can spin up many independent models to compare.

This is precisely the shape of a real ML library. When you soon write `model = LinearRegression()` and `model.fit(...)` in scikit-learn, you now know exactly what kind of object you're creating and what `fit` is doing inside: the very same loop of small downhill steps you built here by hand.